# Notebook 3: Attention and Mixture of Experts (MoE)

In this notebook, we'll dive into how Simply implements two frontier-level capabilities natively in JAX: Attention and MoE. You will examine actual code snippets sourced from `simply.model_lib` and reconstruct their functionality.

Let's add the repo back to our path.

In [10]:
import sys
import os

repo_root = os.path.abspath(os.path.join(os.getcwd(), '.', 'third-party', 'simply'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import jax
import jax.numpy as jnp
import numpy as np
from simply import model_lib
print("Ready for Attention and MoE!")

Ready for Attention and MoE!


## 1. Attention Masks
Most LLMs require a causal mask to prevent sequence tokens from interacting with the future. Look at `model_lib.create_mask`. In JAX, we usually define masks as boolean matrices applied right before softmax.

### Exercise 1: Construct a Causal Mask
Create a boolean square matrix corresponding to sequence length $L=4$ where `True` flags valid connections, using purely JAX numpy arrays (`jnp.tril` or `>=\`).

In [7]:
# Exercise 1 Code

seq_len = 4

# TODO: Using `jnp.arange` or `jnp.tril`, formulate a [seq_len, seq_len] matrix 
# representing a causal mask where rows are querying tokens and columns are key tokens.
causal_mask = jnp.tril(jnp.ones((seq_len, seq_len)))

if causal_mask is not None:
    print(f"Causal Mask:\n{causal_mask}")

Causal Mask:
[[1. 0. 0. 0.]
 [1. 1. 0. 0.]
 [1. 1. 1. 0.]
 [1. 1. 1. 1.]]


## 2. Scaled Dot-Product Attention and Soft-Capping
Simply scales the raw querying interactions prior to executing softmax. To ensure training stability and mitigate numerical explosions, models like Gemma employ *soft-capping* (`soft_cap(x, cap) = cap * tanh(x/cap)`).

### Exercise 2: Implementing Soft-Cap Attention
Calculate the attention logits. Given $Q$, $K$, compute the dot products, scale by $1/\sqrt{D}$, apply a soft-cap of $50.0$, and pass through `jax.nn.softmax` using your mask.

In [12]:
# Exercise 2 Code
from simply.utils import common

D = 16
q = jnp.ones((4, D))
k = jnp.ones((4, D))

# 1. Dot Products and Scaling -> [4, 4]
raw_logits = jnp.einsum('id,jd->ij', q, k) / jnp.sqrt(D) 

# 2. Soft Cap
cap = 50.0
# TODO: Implement taking `cap * jnp.tanh(raw_logits / cap)`
capped_logits = cap * jnp.tanh(raw_logits / cap)

# 3. Softmax with Masking
# Hint: Set elements where mask==False to an extremely small number (`simply.utils.common.neg_inf`)
# before calling jax.nn.softmax.
causal_mask = jnp.tril(jnp.ones((4, 4)))
masked_logits = jnp.where(causal_mask, capped_logits, common.neg_inf(q.dtype))
attention_probs = jax.nn.softmax(masked_logits, axis=-1)

if attention_probs is not None:
    print("Probabilities Sum across keys:", attention_probs.sum(axis=-1))

Probabilities Sum across keys: [1. 1. 1. 1.]


## 3. FeedForward Network Alignment
Review `simply.model_lib.FeedForward`. Notice it is composed of `EinsumLinear` and includes gated activations (like `SwiGLU`) optionally. It maps dimensions: `model_dim` -> `expand_dim` -> `model_dim`.

### Exercise 3: Initialize a Gated Feedforward
Replicate the standard setup parameter list of `simply.model_lib.FeedForward`. Create equations and track weight matrix shapes given a `model_dim` of $16$ and `expand_factor` of 2.

In [13]:
# Exercise 3 Code

# Provide expected shape for each matrix
model_dim = 16
expand_dim = 16 * 2

# TODO: Determine the shape of `ffn_0_weight`, `ffn_0_gate_weight` and `ffn_1_weight`
# if equations are ('...i,io->...o', '...i,io->...o', '...i,io->...o').
expected_shapes = {
    'ffn_0': [model_dim, expand_dim], 
    'ffn_0_gate': [model_dim, expand_dim], 
    'ffn_1': [expand_dim, model_dim]
}

if expected_shapes['ffn_0'] is not None:
    for k,v in expected_shapes.items():
         print(f"{k} param shape expected is {v}")

ffn_0 param shape expected is [16, 32]
ffn_0_gate param shape expected is [16, 32]
ffn_1 param shape expected is [32, 16]


## 4. Sparse Routing in Mixture-of-Experts (MoE)
Simply supports `MoEFeedForward`. Instead of dense MLP operations every token, tokens are routed to experts. A routing linear transformation assigns a probabilistic selection.

### Exercise 4: Router Selection via Top-K
Generate unnormalized router logits for 8 experts across 4 tokens `[4, 8]`. Then, select the top 2 indices for each token. Use `jax.lax.top_k`.

In [15]:
# Exercise 4 Code

router_logits = jax.random.normal(jax.random.PRNGKey(10), (4, 8)) # 4 tokens, 8 experts

# TODO: Obtain `selected_logits` and `selected_indices` representing 
# the top 2 experts for each of the 4 tokens.
k = 2
# topk_outputs = jax.lax.top_k(...)

selected_logits, selected_indices = jax.lax.top_k(router_logits, k=2)

if selected_logits is not None:
    print(f"Selected Expert Indices:\n{selected_indices}")
    print(f"Selected Expert Logits:\n{selected_logits}")

Selected Expert Indices:
[[6 7]
 [1 2]
 [1 2]
 [0 2]]
Selected Expert Logits:
[[0.7985934  0.45149153]
 [1.0553882  0.7867297 ]
 [1.0737875  0.8392266 ]
 [0.5971349  0.20536722]]


## 5. Sparse Softmax and Expert Dispatch
Once tokens identify their assigned experts, they proceed through those specific dense blocks. Simply integrates `jax.lax.ragged_dot` and Megablox, but you can build intuition purely using masked mapping operations.

### Exercise 5: Applying Masks for Experts
Given `selected_indices`, formulate a tensor of shape `[4, 8]` that is `1` if the expert was selected and `0` otherwise, allowing gradient propagation on probability weights. Use one-hot encodings.

In [17]:
jax.nn.one_hot(selected_indices, num_classes=8)

Array([[[0., 0., 0., 0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 0., 0., 0., 1.]],

       [[0., 1., 0., 0., 0., 0., 0., 0.],
        [0., 0., 1., 0., 0., 0., 0., 0.]],

       [[0., 1., 0., 0., 0., 0., 0., 0.],
        [0., 0., 1., 0., 0., 0., 0., 0.]],

       [[1., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 1., 0., 0., 0., 0., 0.]]], dtype=float32)

In [22]:
# Exercise 5 Code

# We have selected_indices from Exercise 4:
# e.g., selected_indices = jnp.array([[2, 5], [0, 7], ...])
selected_indices = jnp.array([[2, 5], [0, 7], [1, 1], [3, 4]])

# TODO: Use `jax.nn.one_hot` on selected_indices corresponding to the total number of experts (8), 
# and aggregate the result across the 'K' dimension (axis 1).
one_hot_indices = jax.nn.one_hot(selected_indices, num_classes=8)
expert_mask = jnp.sum(one_hot_indices, axis=1)

if expert_mask is not None:
    print(f"Expert Boolean Grid:\n{expert_mask}\n")
    print(f"Tokens mapped to Expert 6: {expert_mask[:, 6].sum()}")

Expert Boolean Grid:
[[0. 0. 1. 0. 0. 1. 0. 0.]
 [1. 0. 0. 0. 0. 0. 0. 1.]
 [0. 2. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 1. 0. 0. 0.]]

Tokens mapped to Expert 6: 0.0


---

## ✨ Bonus Material: Additional Topics & Exercises

The following exercises and concepts were merged from the previous `03_attention_and_kv_cache.ipynb` curriculum.

# 03: Attention Mechanisms & Autoregressive KV Cache
This notebook connects the layers to build full attention logic. During token generation (inference), LLMs suffer from recomputation overhead. `simply/model_lib.py` solves this functionally using a Key/Value cache.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('../third-party/simply'))

import jax
import jax.numpy as jnp

## 1. Causal Masking
Before a Transformer can generate the next token, it must ensure that attention can only look 'backwards'. Future tokens cannot influence past ones.
Let's see the heart of the attention loop.

In [26]:
# Simulate sequence positions for Queries and KV pairs
# Shape: [batch_size=1, seq_len=4]
q_positions = jnp.array([[0, 1, 2, 3]])
kv_positions = jnp.array([[0, 1, 2, 3]])
print(f"Q shape: {q_positions.shape}")
print(f"KV shape: {kv_positions.shape}")

# Expand to create causal mask of shape [batch, q_seq, kv_seq]
# 'bl -> bl1' expanding queries, 'bl -> b1l' expanding keys
q_exp = jnp.expand_dims(q_positions, axis=-1)
kv_exp = jnp.expand_dims(kv_positions, axis=-2)
print(f"Q shape: {q_exp.shape}")
print(f"KV shape: {kv_exp.shape}")

causal_mask = q_exp >= kv_exp
print("Causal Matrix:")
# True means unmasked (can attend), False means masked.
print(causal_mask[0].astype(int))
print(f"Causal Mask Shape: {causal_mask.shape}")

Q shape: (1, 4)
KV shape: (1, 4)
Q shape: (1, 4, 1)
KV shape: (1, 1, 4)
Causal Matrix:
[[1 0 0 0]
 [1 1 0 0]
 [1 1 1 0]
 [1 1 1 1]]
Causal Mask Shape: (1, 4, 4)


## 2. Functional KV Caching
In `simply/model_lib.py`, you'll see a function called `updated_decode_state`. Let's understand why it exists.

When generating a token autoregressively, the sequence length is `seq=1`, but the attention must attend to all previous K and V matrices.
Unlike an object that mutates a `.cache` attribute internally, this function takes the raw dictionary `decode_state` and returns an updated *new* dictionary with the new token dynamically inserted at the `cache_position`.

In [28]:
?updated_decode_state

Signature:
updated_decode_state(
    k: jax.Array | numpy.ndarray,
    v: jax.Array | numpy.ndarray,
    segment_positions: jax.Array | numpy.ndarray,
    segment_ids: jax.Array | numpy.ndarray,
    decode_state: Union[NoneType, str, int, float, bool, jax.Array, ForwardRef('AnnotatedArray'), collections.abc.Sequence['PyTree'], collections.abc.Mapping[str, 'PyTree']],
    window_size: int = 0,
    update_kv_cache: bool = True,
) -> tuple[jax.Array | numpy.ndarray, jax.Array | numpy.ndarray, jax.Array | numpy.ndarray, jax.Array | numpy.ndarray, typing.Union[NoneType, str, int, float, bool, jax.Array, ForwardRef('AnnotatedArray'), collections.abc.Sequence['PyTree'], collections.abc.Mapping[str, 'PyTree']]]
Docstring:
Updates decode state when decode_state is not None.

decode_state caches the key, value, segment_positions, segment_ids for all
previous decode steps. The input key, value, segment_positions, segment_ids
will be written at the cache_position in the decode_state caches. decode

In [30]:
from simply.model_lib import updated_decode_state

# Let's mock a fast functional decoding step!
decode_state = {
    'k': jnp.zeros((1, 8, 4, 64)), # [batch, max_seq_len, num_heads, head_dim]
    'v': jnp.zeros((1, 8, 4, 64)),
    'segment_positions': jnp.zeros((1, 8), dtype=jnp.int32),
    'segment_ids': jnp.zeros((1, 8), dtype=jnp.int32),
}

# The new token's key and value:
curr_k = jnp.ones((1, 1, 4, 64))
curr_v = jnp.ones((1, 1, 4, 64))
segment_position = jnp.array([[2]]) # Pretend we are decoding position index 2
segment_ids = jnp.array([[0]])

# Get the updated cache functionally
new_k, new_v, _, _, next_decode_state = updated_decode_state(
    curr_k, curr_v, segment_position, segment_ids, decode_state, window_size=0
)

# Observe the dynamic update! The `k` array has 1s at index 2 now.
print("KV cache at position 1 (empty):", next_decode_state['k'][0, 1, 0, 0])
print("KV cache at position 2 (updated):", next_decode_state['k'][0, 2, 0, 0])

KV cache at position 1 (empty): 0.0
KV cache at position 2 (updated): 1.0


## Exercise:
Inspect the functional attention block `attn` in `simply/model_lib.py`. How does it apply the attention soft cap? Use its implementation to build a dummy attention wrapper.

---

## ✨ Bonus Material: Additional Topics & Exercises

The following exercises and concepts were merged from the previous `04_advanced_architectures.ipynb` curriculum.

# 04: Advanced Architectures - Mixture of Experts (MoE)
As language models grow larger, dense MLPs become extremely expensive. `simply` provides `MoEFeedForward` in `model_lib.py`.

In a Mixture of Experts, a small router network decides which 'experts' (smaller MLPs) process a particular token.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('../third-party/simply'))

import jax
import jax.numpy as jnp

## 1. Top-K Routing Logic
The core component of MoE is computing routing probabilities and selecting the Top-K experts. Let's write the routing equation step-by-step just like in `MoEFeedForward`.

In [ ]:
batch_size, seq_len = 2, 5
num_experts = 8
num_experts_per_token = 2

# Assume the router outputs logits for each expert per token:
router_logits = jax.random.normal(jax.random.PRNGKey(42), (batch_size, seq_len, num_experts))

# Step 1: Get the Top-K logits and their indices
# selected_router_logits shape: [batch, seq, top_k]
selected_router_logits, selected_indices = jax.lax.top_k(router_logits, k=num_experts_per_token)

# Step 2: Normalize the probabilities over only the chosen experts
routing_weights = jax.nn.softmax(selected_router_logits, axis=-1)

print(f"Token (batch 0, pos 0) assigned to experts: {selected_indices[0,0]}")
print(f"With routing weights: {routing_weights[0,0]}")

## Why is this functional implementation fast in JAX?
When JAX compiles `jax.lax.top_k` and the subsequent scattered sparse dot-products (`megablox` or `ragged_dot` as defined in `model_lib.py`), the XLA compiler optimizes the parallel dispatch across experts over TPUs/GPUs.

You can instantiate a full MoE layer in `simply` for your experiments directly:

In [ ]:
from simply.model_lib import MoEFeedForward
from unittest.mock import Mock

# Creating a mock sharding config (since we are testing locally without multiple TPUs)
class DummyConfig:
    activation_partition = None
    ffn0_partition = None
    ffn1_partition = None

moe_layer = MoEFeedForward(
    model_dim=128,
    expand_factor=4,
    sharding_config=DummyConfig(),
    num_experts=4,
    num_experts_per_token=1
)

params = moe_layer.init(jax.random.PRNGKey(0))
x = jnp.ones((2, 5, 128))
inputs_mask = jnp.ones((2, 5), dtype=bool)

# Notice we explicitly pass the params and the mask
outputs, extra = moe_layer.apply(params, x, inputs_mask=inputs_mask)
print("MoE Output shape:", outputs.shape)